# TORP information and multiscale JEPA

This preregistered study separates third-order information from origin dependence, then compares coarse-to-fine multiscale JEPA targets with a matched single-scale recurrence-plot JEPA. Smoke mode is a pipeline check only; no smoke result is evidence for either hypothesis.

## 1. Setup, questions, and preregistered expectations

**T1 — Is a third-order measure useful when pairwise distances cannot distinguish the labels?** The sign diagnostic pairs each z-scored window with its negative. Distances, pairwise cosine similarity, and perimeter TORP are invariant to that flip, while the odd-order generalized cosine changes sign. We expect approximately chance accuracy for D, cosine, perimeter TORP, and sign-invariant RQA/statistics features; gencos TORP, raw attention, and D+series should exceed chance. If the gencos advantage is not observed, the test is inconclusive about its learnability. The direction diagnostic is secondary: forward/reversed windows are irreversible, but reversal also reverses both axes of D.

| Sign-task representation | Preregistered expectation | Reason |
|---|---:|---|
| RP-CNN D; RP-CNN cos; perimeter TORP; sign-invariant RQA/statistics | ~0.50 | Invariant under x → −x |
| gencos TORP; raw attention; RP-CNN D+series | >0.50 | Retains odd-order or signed-series information |

The benchmark tasks `system` and `lyapunov` are not expected to be decisive about D sufficiency; they are an optional extension.

**T2 — Do multiscale JEPA targets improve frozen probes?** Compare RP-CNN (and, by default, RP-S4ND) single-scale JEPA on `RP@0.1+series` with the same encoder trained against recurrence rates `(0.4, 0.2, 0.1, 0.05)`, coarse-to-fine. Each run has one online encoder and one EMA target encoder; the returned model is that single EMA encoder. Probes call the returned encoder with its configured `rp_kind` and `hide=None`, so no cells or states are hidden. Per-level loss columns are monitoring diagnostics; the optimization objective is their mean. Use identical pretraining pools, updates, batches, learning rates, masks, seeds, and probe splits. Report paired-bootstrap test differences per task (multiscale minus single-scale); no direction of effect is assumed.

In [ ]:
from pathlib import Path
import os, sys, time, json, hashlib, warnings
sys.path.insert(0, str(Path('../../src').resolve()))
import numpy as np
import pandas as pd
import torch
from scipy.stats import skew
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, RidgeCV
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from jepa_experiments.data import TASKS
from jepa_experiments.diagnostics import DIAGNOSTIC_TASKS
assert tuple(TASKS) == ('system','lyapunov','noise','changepoint','recall','forecast')
assert tuple(DIAGNOSTIC_TASKS) == ('sign','direction')
from rpjepa.representations import delay_embed, distance_matrix, rp_input, torp
from jepa_experiments.features import rqa_features
from rpjepa.encoders import ENCODERS, make_encoder, n_params, count_flops
from jepa_experiments.train import train_supervised
from jepa_experiments.jepa import pretrain_jepa
from jepa_experiments.probe import embed, linear_probe
from jepa_experiments.experiment import SMOKE, get_split, cached_run, summarise

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
M, TAU, DEPTH = 3, 1, 2
N_T1, D_T1 = (16, 16) if SMOKE else (32, 32)
T1_SIZES = ({'train': 96, 'val': 48, 'test': 96} if SMOKE else
            {'train': 10000, 'val': 2000, 'test': 5000})
FULL_T1_SIZES = {'train': 10000, 'val': 2000, 'test': 5000}
FULL_T1_EPOCHS, FULL_T1_BATCH = 30, 128
FULL_T1_SEEDS = (0, 1, 2)
T1_SEEDS = (0,) if SMOKE else FULL_T1_SEEDS
T1_EPOCHS, T1_PATIENCE = (2, 1) if SMOKE else (30, 5)
T1_BATCH = 8 if SMOKE else 128
PROFILE_N, PROFILE_D, PROFILE_BATCH = 32, 32, FULL_T1_BATCH
RUN_BENCHMARK_TORP = False
N_T2, D_T2 = (16, 16) if SMOKE else (64, 64)
STEPS, BATCH, LR = (20, 4, 1e-3) if SMOKE else (4000, 128, 1e-3)
N_PRE_TASK = 64 if SMOKE else 20000
RECALL_PAIRS = 2 if SMOKE else 4
PROBE_N, VAL_N, LOW_N = (64, 48, 24) if SMOKE else (5000, 2000, 500)
T2_SEEDS = (0,) if SMOKE else (0, 1)
RUN_RP_S4ND = True
BOOTSTRAPS = 2000
CACHE_ROOT = Path('cache/step4') / ('smoke' if SMOKE else '')
TABLE_ROOT = Path('tables/step4') / ('smoke' if SMOKE else '')
CACHE_ROOT.mkdir(parents=True, exist_ok=True); TABLE_ROOT.mkdir(parents=True, exist_ok=True)
print({'device': DEVICE, 'smoke': SMOKE, 'T1_N': N_T1, 'T1_sizes': T1_SIZES, 'T1_seeds': T1_SEEDS,
       'T2_N': N_T2, 'steps': STEPS, 'batch': BATCH, 'pretrain_windows': 6*N_PRE_TASK})

## 2. Information check (no learning)

The TORP perimeter diagonal slice satisfies `perimeter(i,i,k) = 2 D(i,k)`, so it reconstructs D exactly up to floating-point arithmetic. An isometry plus translation preserves D while generally changing both cosine similarity about the origin and gencos; this gives a direct witness that neither origin-dependent quantity is a function of D alone. The sign-flip checks are performed on actual delay-embedded windows. Skewness is reported per diagnostic family on the original (not flipped) windows to document the marginal asymmetry used by T1.

In [ ]:
rng = np.random.default_rng(441)
x_check = torch.as_tensor(rng.normal(size=(1, 24)).astype(np.float32))
s_check = delay_embed(x_check, M, TAU)
D_check = distance_matrix(s_check)
P_check = torp(s_check, 'perimeter')
D_from_perimeter = P_check[:, torch.arange(P_check.shape[1]), torch.arange(P_check.shape[1]), :] / 2
x_neg = -x_check
s_neg = delay_embed(x_neg, M, TAU)
P_neg = torp(s_neg, 'perimeter')
D_neg = distance_matrix(s_neg)
cos_x = rp_input(x_check, M, TAU, 'cos')
cos_neg = rp_input(x_neg, M, TAU, 'cos')
gencos_x = torp(s_check, 'gencos')
gencos_neg = torp(s_neg, 'gencos')
assert torch.allclose(P_check, P_neg, atol=1e-6, rtol=1e-6)
assert torch.allclose(D_from_perimeter, D_check, atol=1e-6, rtol=1e-6)
assert torch.allclose(D_check, D_neg, atol=1e-6, rtol=1e-6)
assert torch.allclose(cos_x, cos_neg, atol=1e-6, rtol=1e-6)
assert torch.allclose(gencos_neg, -gencos_x, atol=1e-6, rtol=1e-6)
assert float(cos_x.min()) >= -1.000001 and float(cos_x.max()) <= 1.000001
assert torch.allclose(cos_x, cos_x.transpose(-1, -2), atol=1e-6, rtol=1e-6)
rp_legacy = rp_input(x_check, M, TAU, 'RP')
rp_tenth = rp_input(x_check, M, TAU, 'RP@0.1')
assert torch.equal(rp_legacy, rp_tenth)

# Distinct coordinates with the same pairwise distances, but different origin-dependent measures.
points = torch.as_tensor(rng.normal(size=(1, 19, 3)).astype(np.float32))
q, _ = torch.linalg.qr(torch.as_tensor(rng.normal(size=(3, 3)).astype(np.float32)))
shift = torch.as_tensor([1.3, -0.7, 0.9], dtype=torch.float32)
isometric_points = points @ q + shift
D_points, D_isometric = distance_matrix(points), distance_matrix(isometric_points)
cos_points = torch.nn.functional.normalize(points, dim=-1) @ torch.nn.functional.normalize(points, dim=-1).transpose(1, 2)
cos_isometric = torch.nn.functional.normalize(isometric_points, dim=-1) @ torch.nn.functional.normalize(isometric_points, dim=-1).transpose(1, 2)
gc_points, gc_isometric = torp(points, 'gencos'), torp(isometric_points, 'gencos')
assert torch.allclose(D_points, D_isometric, atol=1e-6, rtol=1e-6)
assert float((cos_points-cos_isometric).abs().max()) > 1e-3
assert float((gc_points-gc_isometric).abs().max()) > 1e-3

sign_check = get_split('sign', 'probe_test', 300 if SMOKE else 1200, N_T1, M, TAU, base_seed=0)
forward_mask = sign_check.meta.orientation == 'forward'
skew_rows = []
for family, indices in sign_check.meta.loc[forward_mask].groupby('family').groups.items():
    values = skew(sign_check.x[np.asarray(list(indices))], axis=1, bias=False)
    skew_rows.append({'family': family, 'n_forward_windows': len(values), 'mean_skewness': float(np.nanmean(values)),
                      'median_abs_skewness': float(np.nanmedian(np.abs(values)))})
skewness_df = pd.DataFrame(skew_rows).sort_values('family')
information_df = pd.DataFrame([
    {'check':'perimeter diagonal reconstructs D', 'max_abs_error':float((D_from_perimeter-D_check).abs().max()), 'passed':True},
    {'check':'D(x) equals D(-x)', 'max_abs_error':float((D_check-D_neg).abs().max()), 'passed':True},
    {'check':'cos(x) equals cos(-x), symmetric and bounded', 'max_abs_error':float((cos_x-cos_neg).abs().max()), 'passed':True},
    {'check':'perimeter(x) equals perimeter(-x)', 'max_abs_error':float((P_check-P_neg).abs().max()), 'passed':True},
    {'check':'gencos(-x) equals -gencos(x)', 'max_abs_error':float((gencos_neg+gencos_x).abs().max()), 'passed':True},
    {'check':'RP@0.1 equals legacy RP', 'max_abs_error':float((rp_legacy-rp_tenth).abs().max()), 'passed':True},
    {'check':'translated isometry leaves D identical', 'max_abs_error':float((D_points-D_isometric).abs().max()), 'passed':True},
    {'check':'same D but cosine changes under translation', 'max_abs_error':float((cos_points-cos_isometric).abs().max()), 'passed':True},
    {'check':'same D but gencos changes under translation', 'max_abs_error':float((gc_points-gc_isometric).abs().max()), 'passed':True},
])
display(information_df); display(skewness_df)
information_df.to_csv(TABLE_ROOT/'information_check.csv', index=False)
skewness_df.to_csv(CACHE_ROOT/'sign_family_skewness.csv', index=False)

## 3. T1 supervised diagnostic grid

At full scale, train/validation/test sizes are 10,000/2,000/5,000, N=32, d=32, depth=2, seeds 0–2, 30 epochs, patience 5. The primary grid is RP-CNN D, RP-CNN cos, TORP-CNN3D perimeter, TORP-CNN3D gencos, raw attention, and RP-CNN D+series on sign and direction. TORP-S4ND3D perimeter/gencos runs on sign only with seed 0. `RUN_BENCHMARK_TORP=False` optionally adds TORP-CNN3D perimeter/gencos versus RP-CNN D on system and lyapunov, seed 0. All model sizes and forward FLOPs are recorded.

The next cell estimates T1 from one train-and-validation epoch per architecture on 128/48 windows at the full T1 shape (`N=32`, `d=32`, batch 128), before the result loops. Full runs measure these pilots directly; smoke mode uses the recorded full-shape pilot seconds below so that smoke execution stays brief. The estimate scales measured update times by full train/validation update counts and 30 epochs. It excludes data-generation and CPU feature-extraction time. The TORP pilot covers both 3D encoders and both measures.

In [ ]:
T1_SPECS = {
    'rp_cnn_D': ('rp_cnn', {'rp_kind':'D'}),
    'rp_cnn_cos': ('rp_cnn', {'rp_kind':'cos'}),
    'torp_cnn3d_perimeter': ('torp_cnn3d', {'measure':'perimeter'}),
    'torp_cnn3d_gencos': ('torp_cnn3d', {'measure':'gencos'}),
    'raw_attn': ('raw_attn', {}),
    'rp_cnn_D_series': ('rp_cnn', {'rp_kind':'D+series'}),
    'torp_s4nd3d_perimeter': ('torp_s4nd3d', {'measure':'perimeter'}),
    'torp_s4nd3d_gencos': ('torp_s4nd3d', {'measure':'gencos'}),
}
primary_specs = ('rp_cnn_D','rp_cnn_cos','torp_cnn3d_perimeter','torp_cnn3d_gencos','raw_attn','rp_cnn_D_series')
profile_n, profile_val_n, profile_test_n = 128, 48, 48
PROFILE_PILOT_SECONDS = {
    'rp_cnn_D':0.6700965, 'rp_cnn_cos':0.0319565,
    'torp_cnn3d_perimeter':0.3448420, 'torp_cnn3d_gencos':0.1329328,
    'raw_attn':0.0292806, 'rp_cnn_D_series':0.0134339,
    'torp_s4nd3d_perimeter':1.7758811, 'torp_s4nd3d_gencos':0.1619552,
}
profile_times = {}
if SMOKE:
    profile_times = PROFILE_PILOT_SECONDS.copy()
else:
    for spec in primary_specs + ('torp_s4nd3d_perimeter','torp_s4nd3d_gencos'):
        name, kw = T1_SPECS[spec]
        ptrain, pval, ptest = (get_split('sign', split, size, PROFILE_N, M, TAU, base_seed=2901)
                               for split, size in (('train',profile_n),('val',profile_val_n),('test',profile_test_n)))
        pilot = train_supervised(name, PROFILE_D, DEPTH, ptrain, pval, ptest, n_states=PROFILE_N, m=M, tau=TAU,
            epochs=1, patience=1, batch=PROFILE_BATCH, seed=0, device=DEVICE, enc_kw=kw)
        profile_times[spec] = float(pilot['train_s'])
        del pilot
        if torch.cuda.is_available(): torch.cuda.empty_cache()

def estimate_t1_seconds(spec, n_runs):
    pilot_updates = int(np.ceil(profile_n / PROFILE_BATCH) + np.ceil(profile_val_n / PROFILE_BATCH))
    full_updates = int(np.ceil(FULL_T1_SIZES['train'] / FULL_T1_BATCH)
                          + np.ceil(FULL_T1_SIZES['val'] / FULL_T1_BATCH))
    return profile_times[spec] * (full_updates / pilot_updates) * FULL_T1_EPOCHS * n_runs

run_counts = {spec: 0 for spec in profile_times}
for spec in primary_specs: run_counts[spec] = len(('sign','direction')) * len(FULL_T1_SEEDS)
for spec in ('torp_s4nd3d_perimeter','torp_s4nd3d_gencos'): run_counts[spec] = 1
if RUN_BENCHMARK_TORP:
    for spec in ('rp_cnn_D','torp_cnn3d_perimeter','torp_cnn3d_gencos'): run_counts[spec] += 2
estimate_rows = [{'spec':spec,'pilot_train_s':profile_times[spec], 'full_grid_runs':count,
                  'estimated_full_grid_hours':estimate_t1_seconds(spec,count)/3600 if count else 0.0}
                 for spec,count in run_counts.items()]
estimate_df = pd.DataFrame(estimate_rows)
print('T1 estimated wall time from measured N-specific one-epoch pilots (linear workload scaling):',
      round(estimate_df.estimated_full_grid_hours.sum(),2), 'hours; target ≤6 h on an idle GPU.')
display(estimate_df)

def get_t1_data(task, split, seed, n):
    return get_split(task, split, n, N_T1, M, TAU, base_seed=seed, **({'n_pairs':2} if task=='recall' else {}))

def run_t1_one(spec, task, seed):
    name, kw = T1_SPECS[spec]
    tr, va, te = (get_t1_data(task, split, seed, T1_SIZES[split]) for split in ('train','val','test'))
    key = {'study':'step4_t1','implementation_revision':2,'spec':spec,'task':task,'seed':seed,'N':N_T1,'d':D_T1,'depth':DEPTH,
           'sizes':T1_SIZES,'epochs':T1_EPOCHS,'patience':T1_PATIENCE,'batch':T1_BATCH,'m':M,'tau':TAU}
    result = cached_run(CACHE_ROOT/'torp_supervised', key, lambda: train_supervised(
        name, D_T1, DEPTH, tr, va, te, n_states=N_T1, m=M, tau=TAU, epochs=T1_EPOCHS,
        patience=T1_PATIENCE, batch=T1_BATCH, seed=seed, device=DEVICE, enc_kw=kw))
    model = make_encoder(name,D_T1,DEPTH,N_T1,m=M,tau=TAU,**kw).cpu()
    cost = {'params':n_params(model),'flops':count_flops(model,N_T1+(M-1)*TAU,batch=1)}
    return {'spec':spec,'encoder':name,'enc_kw':json.dumps(kw,sort_keys=True),'task':task,'seed':seed,
            'N':N_T1,'d':D_T1,'depth':DEPTH,**result,**cost}

t1_configs = [(spec,task,seed) for task in ('sign','direction') for spec in primary_specs for seed in T1_SEEDS]
t1_configs += [(spec,'sign',0) for spec in ('torp_s4nd3d_perimeter','torp_s4nd3d_gencos')]
if RUN_BENCHMARK_TORP:
    t1_configs += [(spec,task,0) for task in ('system','lyapunov')
                   for spec in ('rp_cnn_D','torp_cnn3d_perimeter','torp_cnn3d_gencos')]
t1_rows = []
for i,(spec,task,seed) in enumerate(t1_configs,1):
    row = run_t1_one(spec,task,seed); t1_rows.append(row)
    print(f'{i}/{len(t1_configs)} {task} {spec} seed={seed} metric={row["metric"]:.4f} train_s={row["train_s"]:.2f}')

# Feature baseline uses only sign-invariant RQA and scalar-statistic columns; the signed final samples are excluded.
rqa_rows = []
for task in ('sign','direction'):
    for seed in T1_SEEDS:
        tr, te = get_t1_data(task,'train',seed,T1_SIZES['train']), get_t1_data(task,'test',seed,T1_SIZES['test'])
        feature_dir = CACHE_ROOT/'diagnostic_features'; feature_dir.mkdir(parents=True,exist_ok=True)
        ftr_path, fte_path = feature_dir/f'{task}_train_{seed}.csv', feature_dir/f'{task}_test_{seed}.csv'
        ftr = pd.read_csv(ftr_path) if ftr_path.exists() else rqa_features(tr.x,m=M,tau=TAU)
        fte = pd.read_csv(fte_path) if fte_path.exists() else rqa_features(te.x,m=M,tau=TAU)
        if not ftr_path.exists(): ftr.to_csv(ftr_path,index=False)
        if not fte_path.exists(): fte.to_csv(fte_path,index=False)
        invariant_cols = [c for c in ftr.columns if not c.startswith('last_')]
        imputer = SimpleImputer(strategy='median')
        ztr = imputer.fit_transform(ftr[invariant_cols]); zte = imputer.transform(fte[invariant_cols])
        result = linear_probe(ztr,tr.y,zte,te.y,'cls')
        rqa_rows.append({'spec':'rqa_invariant_features','encoder':'features','enc_kw':'{}','task':task,
                         'seed':seed,'N':N_T1,'d':np.nan,'depth':np.nan,'metric':result['metric'],
                         'val_metric':np.nan,'params':np.nan,'flops':np.nan,'train_s':np.nan,'epochs_run':np.nan})
t1_df = pd.DataFrame(t1_rows + rqa_rows)
display(t1_df[['spec','task','seed','metric','params','flops','train_s','epochs_run']])
display(summarise(t1_df,['spec','task'],'metric'))
t1_df.to_csv(TABLE_ROOT/'torp_supervised.csv',index=False)
print('T1 grid rows (including RQA feature references):',len(t1_df), '| model runs:',len(t1_rows))

## 4. T2 multiscale JEPA and frozen probes

Full settings match Step 3: an equal unlabeled pool of 20,000 windows from each of the six frozen tasks (120,000 total), 4,000 updates, batch 128, learning rate 1e-3, seeds 0 and 1, N=64, d=64, depth 2. Smoke uses N=16, small pools, 20 updates, and seed 0. The context encoder uses its configured `RP@0.1+series` input for both objectives. The single-scale JEPA target also uses RP@0.1; multiscale EMA targets use rates 0.4 → 0.05, append the preceding level's predicted token set to the next predictor input, and average the level losses.

The full model grid is RP-CNN single-scale/multiscale and, when `RUN_RP_S4ND=True`, RP-S4ND single-scale/multiscale, each at both seeds. Probe evaluations use matched `probe_train`, `probe_val`, and `probe_test` data; primary comparisons use the paired test windows. Low-label probes use 500 training examples.

In [ ]:
SCALES = (0.4,0.2,0.1,0.05)
T2_ENCS = ['rp_cnn'] + (['rp_s4nd'] if RUN_RP_S4ND else [])
T2_CONFIGS = [(enc,obj) for enc in T2_ENCS for obj in ('jepa','jepa_multiscale')]

pretrain_arrays=[]
for task in TASKS:
    difficulty = {'n_pairs':RECALL_PAIRS} if task=='recall' else {}
    pre = get_split(task,'pretrain',N_PRE_TASK,N_T2,M,TAU,base_seed=31415,**difficulty)
    pretrain_arrays.append(pre.x)
x_pre = np.concatenate(pretrain_arrays,axis=0).astype(np.float32,copy=False)
del pretrain_arrays, pre
print('Unlabeled pool:',x_pre.shape,'windows/task:',N_PRE_TASK)
if SMOKE:
    # One-step objective/mask coverage for every JEPA path added in this study.
    smoke_configs = [(enc, mask, 'jepa') for enc in ('rp_cnn','rp_s4nd','rp_vit')
                     for mask in ('future','cell_asym')]
    smoke_configs.append(('rp_cnn','future','mae'))
    pretrain_smoke_rows = []
    for smoke_enc, smoke_mask, smoke_objective in smoke_configs:
        smoke_model, smoke_log = pretrain_jepa(
            smoke_enc, D_T2, DEPTH, x_pre[:4], n_states=N_T2, m=M, tau=TAU,
            mask=smoke_mask, steps=1, batch=2, seed=73, device=DEVICE,
            enc_kw={'rp_kind':'D+series'}, objective=smoke_objective)
        with torch.no_grad():
            smoke_tokens = smoke_model(torch.as_tensor(x_pre[:2], device=DEVICE), hide=None)
        assert smoke_tokens.shape == (2, N_T2, D_T2)
        assert len(smoke_log) == 1 and np.isfinite(smoke_log.loss.iloc[0])
        pretrain_smoke_rows.append({'encoder':smoke_enc,'mask':smoke_mask,
                                    'objective':smoke_objective,'loss':float(smoke_log.loss.iloc[0])})
        del smoke_model
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    print('One-step JEPA smoke coverage:', pretrain_smoke_rows)

def digest_key(key): return hashlib.sha256(json.dumps(key,sort_keys=True,separators=(',',':')).encode()).hexdigest()[:20]
def pretrained_t2(enc,obj,seed):
    enc_kw={'rp_kind':'RP@0.1+series'}
    key={'enc':enc,'objective':obj,'enc_kw':enc_kw,'seed':seed,'N':N_T2,'d':D_T2,'depth':DEPTH,
         'steps':STEPS,'batch':BATCH,'lr':LR,'scales':SCALES if obj=='jepa_multiscale' else None,
         'pretrain_per_task':N_PRE_TASK,'recall_pairs':RECALL_PAIRS,'tasks':list(TASKS)}
    stem=digest_key(key); model_path=CACHE_ROOT/'encoders'/f'{stem}.pt'; log_path=CACHE_ROOT/'logs'/f'{stem}.csv'
    model_path.parent.mkdir(parents=True,exist_ok=True); log_path.parent.mkdir(parents=True,exist_ok=True)
    if model_path.exists() and log_path.exists():
        model=make_encoder(enc,D_T2,DEPTH,N_T2,m=M,tau=TAU,**enc_kw)
        model.load_state_dict(torch.load(model_path,map_location='cpu',weights_only=True)); model.eval()
        return model,pd.read_csv(log_path),True
    t0=time.perf_counter()
    model,log=pretrain_jepa(enc,D_T2,DEPTH,x_pre,n_states=N_T2,m=M,tau=TAU,
        steps=STEPS,batch=BATCH,lr=LR,seed=seed,device=DEVICE,enc_kw=enc_kw,objective=obj,
        scales=SCALES)
    elapsed=time.perf_counter()-t0
    log['train_s']=elapsed
    torch.save({k:v.detach().cpu() for k,v in model.state_dict().items()},model_path)
    log.to_csv(log_path,index=False)
    return model,log,False

models={}; pretrain_rows=[]; loss_rows=[]
for enc,obj in T2_CONFIGS:
    for seed in T2_SEEDS:
        model,log,loaded=pretrained_t2(enc,obj,seed); models[(enc,obj,seed)]=model
        log=log.copy(); log['encoder']=enc; log['objective']=obj; log['seed']=seed; loss_rows.append(log)
        row={'encoder':enc,'objective':obj,'seed':seed,'steps':STEPS,'cache_loaded':loaded,
             'train_s':float(log.train_s.iloc[0]),
             'initial_emb_std':float(log.emb_std.iloc[0]),'final_emb_std':float(log.emb_std.iloc[-1]),
             'collapse_flag':bool(log.emb_std.iloc[-1] < 0.05*log.emb_std.iloc[0])}
        if obj=='jepa_multiscale':
            for level in range(len(SCALES)):
                row[f'loss_level_{level}_first']=float(log[f'loss_level_{level}'].iloc[0])
                row[f'loss_level_{level}_last']=float(log[f'loss_level_{level}'].iloc[-1])
        pretrain_rows.append(row)
        print(f'{enc} {obj} seed={seed}: loss={log.loss.iloc[0]:.4f}→{log.loss.iloc[-1]:.4f}, emb_std={log.emb_std.iloc[-1]:.4f}')
pretrain_df=pd.DataFrame(pretrain_rows); log_df=pd.concat(loss_rows,ignore_index=True)
if SMOKE:
    ms=pretrain_df[pretrain_df.objective=='jepa_multiscale']
    for _,row in ms.iterrows():
        for level in range(len(SCALES)):
            assert row[f'loss_level_{level}_last'] < row[f'loss_level_{level}_first'], f'level {level} loss did not decrease'
        assert row.final_emb_std > 0, 'multiscale embedding collapsed'
pretrain_df.to_csv(TABLE_ROOT/'multiscale_pretrain.csv',index=False)

def metric_prediction(y,pred,kind):
    y=np.asarray(y); pred=np.asarray(pred)
    if kind=='cls': return float(np.mean(pred==y))
    yy=y.astype(float); pp=np.asarray(pred,dtype=float); denom=np.square(yy-yy.mean()).sum()
    return float(1-np.square(yy-pp).sum()/denom) if denom>0 else np.nan
def fit_predict(Ztr,ytr,Zte,kind):
    scaler=StandardScaler(); A=scaler.fit_transform(Ztr); B=scaler.transform(Zte)
    if kind=='cls':
        cv=StratifiedKFold(n_splits=3,shuffle=True,random_state=0)
        model=GridSearchCV(LogisticRegression(max_iter=2000,solver='lbfgs',random_state=0),
            {'C':np.logspace(-3,2,6)},cv=cv,scoring='accuracy',refit=True,n_jobs=1)
        model.fit(A,ytr); return model.predict(B)
    model=RidgeCV(alphas=np.logspace(-3,3,13)); model.fit(A,np.asarray(ytr,dtype=float)); return model.predict(B)

def probe_data(task,split,seed,n):
    difficulty={'n_pairs':RECALL_PAIRS} if task=='recall' else {}
    return get_split(task,split,n,N_T2,M,TAU,base_seed=seed,**difficulty)
probe_rows=[]; prediction_store={}
for enc,obj in T2_CONFIGS:
    method=f'{enc}_{obj}'
    for seed in T2_SEEDS:
        model=models[(enc,obj,seed)]
        for task in TASKS:
            tr=probe_data(task,'probe_train',seed,PROBE_N)
            # Batch 128: at 512 the RP-S4ND activations exceed the 12 GB GPU and spill to system memory.
            ztr=embed(model,tr.x,batch=32 if SMOKE else 128,device=DEVICE)
            for split,n_eval in (('probe_val',VAL_N),('probe_test',PROBE_N)):
                te=probe_data(task,split,seed,n_eval)
                zte=embed(model,te.x,batch=32 if SMOKE else 128,device=DEVICE)
                result=linear_probe(ztr,tr.y,zte,te.y,tr.kind)
                pred=fit_predict(ztr,tr.y,zte,tr.kind)
                probe_rows.append({'method':method,'encoder':enc,'objective':obj,'task':task,'seed':seed,
                    'split':split,'n_train':len(tr.x),'n_eval':len(te.x),**result})
                if split=='probe_test': prediction_store[(method,task,seed)]=(te.y.copy(),np.asarray(pred).copy(),tr.kind)
            chosen=np.random.default_rng(9000+seed).choice(len(tr.x),size=min(LOW_N,len(tr.x)),replace=False)
            low=linear_probe(ztr[chosen],tr.y[chosen],zte,te.y,tr.kind)
            probe_rows.append({'method':method,'encoder':enc,'objective':obj,'task':task,'seed':seed,
                'split':'lowlabel_test','n_train':len(chosen),'n_eval':len(te.x),**low})
probe_df=pd.DataFrame(probe_rows)
probe_df.to_csv(TABLE_ROOT/'multiscale_probes.csv',index=False)
probe_summary=summarise(probe_df,['encoder','objective','task','split'],'metric')
display(probe_summary.round(4))

def paired_bootstrap(method_a,method_b,task,n_resamples=BOOTSTRAPS):
    seed_diffs=[]; boot_by_seed=[]
    for seed in T2_SEEDS:
        a=prediction_store[(method_a,task,seed)]; b=prediction_store[(method_b,task,seed)]
        ya,pa,kind=a; yb,pb,_=b
        if not np.array_equal(ya,yb): raise ValueError(f'unpaired probe windows for {task}, seed {seed}')
        seed_diffs.append(metric_prediction(ya,pa,kind)-metric_prediction(yb,pb,kind))
        rng=np.random.default_rng(12345+seed); reps=[]; n=len(ya)
        for offset in range(0,n_resamples,100):
            size=min(100,n_resamples-offset); ix=rng.integers(0,n,size=(size,n))
            if kind=='cls':
                delta=(pa==ya).astype(float)-(pb==yb).astype(float); reps.extend(delta[ix].mean(axis=1))
            else:
                yy=ya.astype(float); p1=np.asarray(pa,dtype=float); p2=np.asarray(pb,dtype=float)
                ys=yy[ix]; means=ys.mean(axis=1,keepdims=True); denom=np.square(ys-means).sum(axis=1)
                numer=np.square(ys-p2[ix]).sum(axis=1)-np.square(ys-p1[ix]).sum(axis=1)
                reps.extend(np.divide(numer,denom,out=np.full(size,np.nan),where=denom>0))
        boot_by_seed.append(np.asarray(reps,dtype=float))
    boot=np.nanmean(np.stack(boot_by_seed),axis=0)
    return {'difference':float(np.mean(seed_diffs)),
        'seed_sd':float(np.std(seed_diffs,ddof=1)) if len(seed_diffs)>1 else np.nan,
        'ci_low':float(np.nanpercentile(boot,2.5)),'ci_high':float(np.nanpercentile(boot,97.5)),
        'seeds':len(seed_diffs),'bootstrap_replicates':n_resamples}
comparison_rows=[]
for enc in T2_ENCS:
    ms=f'{enc}_jepa_multiscale'; ss=f'{enc}_jepa'
    for task in TASKS:
        comparison_rows.append({'encoder':enc,'task':task,'method_a':ms,'method_b':ss,
                               **paired_bootstrap(ms,ss,task)})
comparison_df=pd.DataFrame(comparison_rows)
comparison_df.to_csv(TABLE_ROOT/'multiscale_comparison.csv',index=False)
display(comparison_df)
print('T2 full grid:',len(T2_ENCS)*2*len(T2_SEEDS),'pretraining runs; probe rows:',len(probe_df))

## 5. Outputs and run accounting

`information_check.csv`, `torp_supervised.csv`, `multiscale_probes.csv`, and `multiscale_comparison.csv` are written to `../../results/tables/jepa/step4/`; smoke tables are isolated under `smoke/`. Cached data, features, checkpoints, and logs are under `../../data/processed/jepa/step4/` with a separate smoke subfolder where applicable. The full T1 primary grid has 36 seeded model runs plus two TORP-S4ND3D runs; optional benchmark TORP adds six runs. The T2 full grid has 8 pretraining runs with RP-S4ND enabled (4 if disabled). See the measured-pilot estimate above for T1; T2 training workload is `runs × 4000` updates, each at batch 128.

## 6. Findings

Full run. T1: N = 32, d = 32, train/val/test 10,000/2,000/5,000, seeds 0–2 (TORP-S4ND3D: seed 0 only). T2: N = 64, d = 64, 4,000 updates, seeds 0–1. Accuracy on test; chance = 0.5.

**Information check (section 2): passed exactly.**
- The perimeter TORP rebuilds D with zero error, so it contains nothing beyond D.
- D, pairwise cosine and the perimeter TORP are identical for x and −x, while gencos flips sign.
- A translated rotation leaves D unchanged (max error 5 × 10⁻⁷) but changes cosine by 1.71 and gencos by 2.0. So origin-dependent measures carry information D does not.

**T1: the pre-registered pattern holds.**

| representation | sign | direction | params | FLOPs / window |
|---|---|---|---|---|
| RP-CNN on D | 0.500 (all seeds) | 0.501 | 35.7 k | 72 M |
| RP-CNN on pairwise cosine | 0.500 (all seeds) | 0.70 (0.80, 0.81, 0.50) | 35.7 k | 72 M |
| TORP-CNN3D perimeter | 0.500 (all seeds) | 0.500 | 1.6 k | 71 M |
| TORP-CNN3D gencos | **0.76** (0.88, 0.89, 0.50) | 0.72 (0.82, 0.83, 0.50) | 1.6 k | 72 M |
| TORP-S4ND3D perimeter / gencos (seed 0) | 0.500 / 0.676 | – | 1.5 k | 25 M |
| RQA + statistics features (sign-invariant) | 0.500 | 0.500 | – | – |
| raw attention | 0.998 | 0.837 | 26.7 k | 1.1 M |
| RP-CNN on D + series | 0.997 | 0.837 | 35.8 k | 72 M |

1. **A TORP adds information only through an origin-dependent, odd-order triple measure.** Every representation that is a function of D sits at exactly 0.500 on `sign`: D itself, the perimeter TORP and the RQA features. Pairwise cosine is origin-dependent but even-order, and it also sits at 0.500. Only gencos, which is third-order and odd, gets above chance (0.88–0.89 on two of three seeds). This separates the "third order" claim from "origin dependence": on `sign`, the third-order term is what matters.
2. **But a TORP is not the cheapest way to get this information.** The raw series, or D plus the two series channels, solves `sign` almost perfectly (0.997–0.998) at equal or much lower cost.
3. **D also loses time direction.** Reversing a window reverses each delay vector, which preserves Euclidean distances. So D of the reversed window is D rotated by 180°, and D-only models and RQA features score 0.50 on `direction`. Cosine and gencos (0.80–0.83 on working seeds) and the series-based models (0.837) recover it.
4. **Origin-dependent encoders are unstable.** In 3 of 9 cosine/gencos runs, the model stayed at exactly 0.5, and early stopping (patience 5) ended it at epoch 6.

**T2: multiscale JEPA targets did not help.** Mean frozen-probe metric over the six tasks:
- RP-CNN: single-scale 0.510, multiscale 0.504.
- RP-S4ND: single-scale 0.407, multiscale 0.406.

Paired differences per task are within ±0.02 and mostly not significant. The exceptions are a loss on noise for RP-CNN (−0.042, 95 % −0.046 to −0.037) and a gain on lyapunov for RP-S4ND (+0.018). With 500 labels the picture is the same (0.440 vs 0.438). No run collapsed. The coarse-to-fine targets across recurrence rates (0.4 → 0.05) give the encoder no information the single-rate target lacks, at about twice the pretraining cost.

**Caveats.**
- To make N³ inputs feasible, the TORP encoders were narrowed to 8 (CNN3D) and 4 (S4ND3D) internal channels, about 1.6 k parameters vs 36 k for RP-CNN. Their accuracies are lower bounds.
- Engineering detail: TORP-CNN3D went from 29.8 s to 0.12 s per training batch, and TORP-S4ND3D from > 330 s to 2.3 s (N = 32, batch 128, deterministic cuDNN).
- Mean skewness of the sign task's families: logistic −0.14, Hénon −0.59, GARCH |r| +0.86.
- Two or three seeds; one pretraining schedule.
